# Calculate Tumor Surface Curvature

Extracts surface area, watertightness, and Gaussian curvature of the Enhancing Tumor (ET) subregion from BraTS 2023 using marching cubes + trimesh. Outputs `Smoothed_Curvature_Analysis_ET.pkl`.


In [ ]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join
import pickle as pkl
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd


import nibabel as nib

from skimage import feature
from scipy import ndimage as ndi
from skimage.feature import shape_index
from skimage.draw import disk
from skimage import measure



import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from scipy.ndimage import gaussian_filter




import trimesh
from trimesh.curvature import discrete_gaussian_curvature_measure, discrete_mean_curvature_measure, sphere_ball_intersection




import warnings
warnings.simplefilter("ignore")

# to interact  with plot

In [ ]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = patient_id + '/' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    sample_filename1 = baseloc + pefix + suffixs[0]
    sample_img1_f = nib.load(sample_filename1)
    sample_img1 = np.asarray(sample_img1_f.dataobj)
    # sample_img1 = np.rot90(sample_img1)

    sample_filename2 = baseloc + pefix + suffixs[1]
    sample_img2_f = nib.load(sample_filename2)
    sample_img2 = np.asarray(sample_img2_f.dataobj)
    # sample_img2  = np.rot90(sample_img2)

    sample_filename3 = baseloc + pefix + suffixs[2]
    sample_img3_f = nib.load(sample_filename3)
    sample_img3 = np.asarray(sample_img3_f.dataobj)
    # sample_img3  = np.rot90(sample_img3)

    sample_filename4 = baseloc + pefix + suffixs[3]
    sample_img4_f = nib.load(sample_filename4)
    sample_img4 = np.asarray(sample_img4_f.dataobj)
    # sample_img4  = np.rot90(sample_img4)

    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
    
    return sample_img1, sample_img2, sample_img3, sample_img4, sample_mask 

def calculate_bounding_box(data):
    """
    Calculate the bounding box for the region of interest in a 3D MRI mask file.

    :param data: The MRI image mask.
    :return: roi_start, roi_end coordinates.
    """

    # Find the indices where the tumor is present
    indices = np.array(np.where(data == 1))

    # Calculate the bounding box
    roi_start = np.min(indices, axis=1)
    roi_end = np.max(indices, axis=1) + 1  # Add 1 to include the end index

    return tuple(roi_start), tuple(roi_end)


def crop_mri_mask(data, roi_start, roi_end):
    """
    Crop a region of interest from a 3D MRI mask file.

    :param data: The MRI image mask.
    :param roi_start: The start coordinates (x, y, z) of the ROI.
    :param roi_end: The end coordinates (x, y, z) of the ROI.
    :return: Cropped MRI data.
    """
    # Crop the data
    cropped_data = data[roi_start[0]:roi_end[0], roi_start[1]:roi_end[1], roi_start[2]:roi_end[2]]

    return cropped_data

def calculate_curvature(dataset, patient_id, spacing=1, smooth=False):
    
    sample_img1, sample_img2, sample_img3, sample_img4, mask  = read_MRI(dataset, patient_id)
    masks = preprocess_mask_labels(mask)
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]
    
    if smooth:
        mask_WT = gaussian_filter(mask_ET, sigma=0.8)
#         roi_start, roi_end = calculate_bounding_box(mask_WT)
#         cropped_mri = crop_mri_mask(smoothed_mask_WT, roi_start, roi_end)
    else:
        mask_WT = mask_ET
#         roi_start, roi_end = calculate_bounding_box(mask_WT)
#         cropped_mri = crop_mri_mask(mask_WT, roi_start, roi_end)
    
    
    # marching_cubes extracts an isosurface mesh from the binary mask
    verts, faces, _, _ = measure.marching_cubes(volume=mask_WT,
                                                spacing = (1*spacing,1*spacing,1*spacing),
                                                allow_degenerate=False,
                                                method='lewiner', 
                                                step_size=1)
    surface_area = measure.mesh_surface_area(verts, faces)
    
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
    watertight = mesh.is_watertight
    
    gaussian_curvature = trimesh.curvature.discrete_gaussian_curvature_measure(mesh, mesh.vertices, 1)
    # Round to 2dp to reduce noise from floating-point mesh normals
    gaussian_curvature = gaussian_curvature.round(2)
#     mean_curvature = trimesh.curvature.discrete_mean_curvature_measure(mesh, mesh.vertices, 1)

    # Count faces belonging to flat regions (missing from curved facets)
    flatten_facets = []
    for _facets in mesh.facets:
        for _facet in _facets:
            flatten_facets.append(_facet)
            
    total_faces = len(faces)
    missing_faces = total_faces - len(flatten_facets)
#     print(missing_faces, total_faces, len(flatten_facets))
    
    
    return {'surface_area': surface_area, 
            'watertight': watertight, 
            'gaussian_curvature': gaussian_curvature, 
            'total_faces': total_faces, 
            'missing_faces': missing_faces}

In [ ]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

curvature_analysis = {}
count = 0
for _id in patient_ids:
    try:
        if count%30 == 0:
            print('Done:', count)
        count += 1

        # smooth=True applies Gaussian filter (sigma=0.8) before marching cubes
        curvature = calculate_curvature(dataset, _id, spacing=1, smooth=True)
        curvature_analysis[_id] = curvature
    except:
        continue

In [ ]:
with open('../Results/Analysis_Results/Smoothed_Curvature_Analysis_ET.pkl', 'wb') as handle:
    pkl.dump(curvature_analysis, handle, protocol=pkl.HIGHEST_PROTOCOL)